# 03 — Geographic Maps

**Project:** Private and Public Good Provision of Mental Health Services in France  

---

## Overview

This notebook produces all choropleth maps used in the paper's geographic analysis section. Three maps are generated:

1. **Private therapist density** — number of liberal therapists per 100,000 inhabitants, by department.
2. **CMP density** — number of *Centres Médico-Psychologiques* (public mental health outpatient centres) per 100,000 inhabitants, by department.
3. **Most and least offered psychological services** — chloropleth maps coloured by the dominant (or least common) service specialty in each department.

Visually comparing maps 1 and 2 reveals the geographic substitution pattern at the core of our research question: areas with fewer private therapists tend to have denser CMP coverage.

**Inputs:**
- `data/processed/full_therapists.geojson` — scraped therapist records
- `data/raw/departements.geojson` — France department polygons (IGN/OpenStreetMap)
- `data/raw/POPULATION_MUNICIPALE_DEPARTEMENT_FRANCE.xlsx` — INSEE 2021 population figures
- `data/raw/data-2.xlsx` — DREES CMP density data (2019)
- `data/raw/cities.csv` — French city coordinates for label placement
- `data/processed/min_max_therapist.xlsx` — service specialty summary from notebook 02

**Outputs:** saved to `figures/`

---

## Setup

In [ ]:
import os
import numpy as np
import unicodedata
import textwrap

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import mapclassify
import seaborn as sns

from shapely.geometry import Point
from shapely.wkt import loads

os.makedirs("figures", exist_ok=True)

---

## 1. Load Shared Geographic Assets

Several datasets are shared across all maps. We load them once here:
- **Department polygons** — the base layer for all choropleths.
- **Population data** — used to normalise therapist and CMP counts to per-capita rates.
- **City coordinates** — used to annotate major regional capitals on each map.

In [ ]:
# France department polygons
departments_geo = gpd.read_file("data/raw/departements.geojson", low_memory=True)
departments_geo["code"] = departments_geo["code"].astype(str)

# INSEE 2021 municipal population by department
df_pop = pd.read_excel(
    "data/raw/POPULATION_MUNICIPALE_DEPARTEMENT_FRANCE.xlsx",
    sheet_name="III_1_insee_population_fr_depar"
)
population_2021 = df_pop[["dep", "p21_pop", "superf"]]

# CMP density data from DREES (2019 figures)
df_cmp = pd.read_excel("data/raw/data-2.xlsx", skiprows=3)
df_cmp = df_cmp[["Code", "Libellé", "Densité de CMP 2019"]]
df_cmp["Code"] = df_cmp["Code"].astype(str)

print("Departments:", len(departments_geo))
print("Population records:", len(population_2021))
print("CMP records:", len(df_cmp))

In [ ]:
# Major regional capitals for map annotation
# We normalise city names (remove accents, lowercase) to match the cities CSV format
def strip_accents(s):
    return unicodedata.normalize("NFD", s).encode("ascii", "ignore").decode("utf-8").lower()

capitals = [
    "Paris", "Orleans", "Dijon", "Rouen", "Lille", "Strasbourg", "Nantes",
    "Rennes", "Bordeaux", "Toulouse", "Lyon", "Marseille", "Ajaccio",
    "Amiens", "Metz", "Chalons en champagne", "Clermont ferrand",
    "Besançon", "Caen", "Montpellier"
]
capitals_normalised = [strip_accents(c) for c in capitals]

villes = pd.read_csv("data/raw/cities.csv")
villes_chefs = villes[villes["label"].isin(capitals_normalised)].drop_duplicates(subset="label")
villes_chefs["geometry"] = villes_chefs.apply(
    lambda row: Point(row["longitude"], row["latitude"]), axis=1
)
villes_chefs_gdf = gpd.GeoDataFrame(villes_chefs, geometry="geometry", crs="EPSG:4326")

print(f"City annotations: {len(villes_chefs_gdf)} cities loaded")

### Helper — City Annotation Layer

We define a reusable function to overlay city labels and white dot markers on any choropleth axis.

In [ ]:
def annotate_cities(ax, cities_gdf):
    """Overlay city name labels and white dot markers on a matplotlib axis."""
    for _, row in cities_gdf.iterrows():
        label = row["label"].title().replace(" ", "\n")
        ax.text(
            row.geometry.x + 0.1,
            row.geometry.y + 0.1,
            label,
            fontsize=10,
            ha="center",
            color="black",
            family="sans-serif",
            weight=600,
            bbox=dict(facecolor="white", alpha=0.5, edgecolor="none", boxstyle="round,pad=0.3")
        )
        ax.scatter(row.geometry.x, row.geometry.y, color="white", s=40, edgecolor="black", zorder=3)

---

## 2. Map 1 — Private Therapist Density

We aggregate the therapist dataset to the department level, merge in population data, and compute a per-capita rate (therapists per 100,000 inhabitants). Departments with no therapists listed on Psychologue.net are shown in white.

The map uses a sequential red colour scale: darker shades indicate higher therapist availability.

In [ ]:
# Load therapist data and aggregate to department level
therapists = gpd.read_file("data/processed/full_therapists.geojson", low_memory=True)

dept_stats = (
    therapists
    .groupby("INSEE_DEP")
    .agg(num_therapists=("Name", "nunique"))
    .reset_index()
)

dept_stats = dept_stats.merge(
    population_2021[["dep", "p21_pop", "superf"]],
    left_on="INSEE_DEP", right_on="dep", how="left"
).drop(columns=["dep"])

dept_stats["therapists_per_100k"] = dept_stats["num_therapists"] / dept_stats["p21_pop"] * 100_000
dept_stats["INSEE_DEP"] = dept_stats["INSEE_DEP"].astype(str)

doc_map = departments_geo.merge(
    dept_stats, left_on="code", right_on="INSEE_DEP", how="left"
)

print(f"Departments with therapist data: {dept_stats['INSEE_DEP'].nunique()}")
print(f"Range: {dept_stats['therapists_per_100k'].min():.1f} – {dept_stats['therapists_per_100k'].max():.1f} per 100k")

In [ ]:
fig, ax = plt.subplots(figsize=(25, 12))

doc_map.plot(
    column="therapists_per_100k",
    cmap="Reds",
    linewidth=0.8,
    ax=ax,
    edgecolor="black",
    legend=True,
    legend_kwds={
        "label": "Médecins pour 100 000 habitants",
        "orientation": "vertical",
        "shrink": 0.7
    },
    missing_kwds={"color": "white", "label": "No Data"},
)

annotate_cities(ax, villes_chefs_gdf)

ax.set_title(
    "Distribution of Private Therapists (per 100,000 inhabitants)",
    fontsize=20, fontweight="bold"
)
ax.axis("off")

plt.tight_layout()
plt.savefig("figures/map_therapist_density.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: figures/map_therapist_density.png")

---

## 3. Map 2 — CMP Density

We map the DREES 2019 CMP density figures (CMPs per 100,000 inhabitants) using the same colour scale and annotation layer as Map 1. Directly comparing the two maps reveals which departments rely more heavily on public versus private provision.

In [ ]:
cmp_map = departments_geo.merge(df_cmp, left_on="code", right_on="Code", how="left")

fig, ax = plt.subplots(figsize=(25, 12))

cmp_map.plot(
    column="Densité de CMP 2019",
    cmap="Reds",
    linewidth=0.8,
    ax=ax,
    edgecolor="black",
    legend=True,
    legend_kwds={
        "label": "CMP pour 100 000 habitants",
        "orientation": "vertical",
        "shrink": 0.7
    },
    missing_kwds={"color": "white", "label": "No Data"},
)

annotate_cities(ax, villes_chefs_gdf)

ax.set_title(
    "Distribution of Centres Médico-Psychologiques (per 100,000 inhabitants, 2019)",
    fontsize=20, fontweight="bold"
)
ax.axis("off")

plt.tight_layout()
plt.savefig("figures/map_cmp_density.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: figures/map_cmp_density.png")

---

## 4. Maps 3 & 4 — Most and Least Offered Services by Department

These maps colour each department by its most (or least) prevalent therapy specialty, as computed in notebook 02. Service categories are encoded as integers for colour mapping purposes.

**Map 3:** Most offered service — nearly all departments list Depression or Psychotherapy as their top specialty, indicating a concentration of the private sector in general-purpose therapeutic needs.

**Map 4:** Least offered service — reveals which niche conditions are systematically underserved by private therapists. Many of these (e.g. Autism, Anorexia, Addictions) are traditionally handled by CMPs, pointing toward complementarity rather than pure substitution.

### 4.1 — Load and Prepare Service Data

In [ ]:
min_max = pd.read_excel("data/processed/min_max_therapist.xlsx")
departements = gpd.read_file("data/raw/departements.geojson")
departements["code"] = departements["code"].astype(str)

# --- Most-offered service encoding ---
most_mapping = {
    "Dépression": 1,
    "Psychothérapeute": 2,
    "Anxiété": 3,
    "Formation étude": 4
}
min_max["most_treated_service_id"] = min_max["most_treated_service"].map(most_mapping)

# --- Least-offered service encoding ---
least_mapping = {
    "Addiction au sexe": 1, "Autisme": 2, "Boulimie": 3, "Analyse des pratiques": 4,
    "Affirmation de soi": 5, "Anorexie": 6, "Coaching": 7, "Deuil": 8,
    "Burn-out": 9, "Ateliers": 10, "Addictions": 11, "Analyse transactionnelle": 12,
    "Aptitudes sociales": 13, "Addiction aux jeux": 14, "Hypersensibilité": 15,
    "Alcoolisme": 16, "Adoption": 17, "Dépendance affective": 18,
    "Agressivité": 19, "Achat compulsif": 20, "Anxiété": 21, "Formation étude": 22
}
min_max["least_treated_service_id"] = min_max["least_treated_service"].map(least_mapping)

# English translations for the least-offered map legend
service_translation = {
    "Addiction au sexe": "Sex Addiction", "Autisme": "Autism", "Boulimie": "Bulimia",
    "Analyse des pratiques": "Practice Analysis", "Affirmation de soi": "Self-Assertion",
    "Anorexie": "Anorexia", "Coaching": "Coaching", "Deuil": "Grief",
    "Burn-out": "Burnout", "Ateliers": "Workshops", "Addictions": "Addictions",
    "Analyse transactionnelle": "Transactional Analysis", "Aptitudes sociales": "Social Skills",
    "Addiction aux jeux": "Gaming Addiction", "Hypersensibilité": "Hypersensitivity",
    "Alcoolisme": "Alcoholism", "Adoption": "Adoption",
    "Dépendance affective": "Emotional Dependence", "Agressivité": "Aggressiveness",
    "Achat compulsif": "Compulsive Buying", "Anxiété": "Anxiety",
    "Formation étude": "Study Training"
}
min_max["least_treated_service_en"] = min_max["least_treated_service"].map(service_translation)

# Merge with department geometry
min_max["INSEE_DEP"] = min_max["INSEE_DEP"].astype(str)
geo = min_max.merge(departements, how="left", left_on="INSEE_DEP", right_on="code")
geo.drop(columns=["Unnamed: 0"], errors="ignore", inplace=True)

# Convert geometry column if stored as WKT strings
geo["geometry"] = geo["geometry"].apply(lambda x: loads(x) if isinstance(x, str) else x)
geo = gpd.GeoDataFrame(geo, geometry="geometry", crs="EPSG:4326")

# Exclude overseas departments (Martinique, Guadeloupe, etc.)
overseas = ["971", "972", "973", "974", "976"]
geo = geo[~geo["INSEE_DEP"].isin(overseas)].reset_index(drop=True)

print(f"Departments in map dataset: {len(geo)}")

### 4.2 — City Annotation for Service Maps

For the service maps we annotate using centroid-based coordinates (Lambert 93 projection) for precise placement, then convert back to WGS84 for plotting.

In [ ]:
# City-to-department code mapping for annotation
city_dept_map = {
    "75": "Paris", "69": "Lyon", "13": "Marseille", "31": "Toulouse",
    "33": "Bordeaux", "44": "Nantes", "67": "Strasbourg", "59": "Lille",
    "35": "Rennes", "45": "Orleans", "21": "Dijon", "63": "Clermont-Ferrand",
    "80": "Amiens", "57": "Metz", "14": "Caen", "25": "Besançon",
    "76": "Rouen", "2A": "Ajaccio", "34": "Montpellier", "51": "Châlons-en-Champagne"
}

# Reproject to Lambert 93 for accurate centroid computation
geo_l93 = geo.to_crs(epsg=2154)
geo_l93["centroid_x"] = geo_l93.geometry.centroid.x
geo_l93["centroid_y"] = geo_l93.geometry.centroid.y
geo_l93["big_city_name"] = geo_l93["code"].map(city_dept_map)

big_cities = geo_l93[geo_l93["big_city_name"].notna()]

### 4.3 — Map 3: Most Offered Service

In [ ]:
most_colors = {
    1: "#377EB8",  # Depression — Blue
    2: "#F0E442",  # Psychotherapy — Yellow
    3: "#E69F00",  # Anxiety — Orange
    4: "#009E73",  # Study Training — Green
}

geo["most_color"] = geo["most_treated_service_id"].map(most_colors)

fig, ax = plt.subplots(figsize=(12, 10))

geo.plot(
    color=geo["most_color"],
    linewidth=0.5,
    edgecolor="black",
    ax=ax
)

# Annotate major cities using WGS84 coordinates
cities_wgs84 = {
    "Paris": (2.3522, 48.8566), "Lyon": (4.85, 45.75), "Marseille": (5.3698, 43.2965),
    "Toulouse": (1.4442, 43.6045), "Bordeaux": (-0.5792, 44.8378), "Nantes": (-1.5536, 47.2184),
    "Strasbourg": (7.7521, 48.5734), "Lille": (3.0573, 50.6292), "Rennes": (-1.6778, 48.1173),
    "Orleans": (1.9093, 47.9029), "Dijon": (5.0415, 47.3220), "Clermont Ferrand": (3.0870, 45.7772),
    "Amiens": (2.3022, 49.8950), "Metz": (6.1757, 49.1193), "Caen": (-0.3707, 49.1829),
    "Besancon": (6.0241, 47.2378), "Rouen": (1.0999, 49.4432), "Ajaccio": (8.7386, 41.9192),
    "Montpellier": (3.8767, 43.6108), "Chalons En Champagne": (4.3673, 48.9581)
}

for city, (lon, lat) in cities_wgs84.items():
    ax.scatter(lon, lat, color="white", edgecolor="black", s=60, zorder=3)
    ax.text(
        lon + 0.2, lat + 0.2,
        textwrap.fill(city, width=10),
        fontsize=8, fontweight="bold", color="black",
        ha="center", va="center",
        bbox=dict(facecolor="white", alpha=0.7, edgecolor="none", boxstyle="round,pad=0.3")
    )

legend_patches = [
    mpatches.Patch(color=most_colors[1], label="Depression"),
    mpatches.Patch(color=most_colors[2], label="Psychotherapy"),
]
ax.legend(handles=legend_patches, title="Most Offered Service", loc="lower left", fontsize=10)

ax.set_aspect(1.3)
ax.set_xlim(-5, 10)
ax.set_ylim(41, 52)
ax.axis("off")

plt.tight_layout()
plt.savefig("figures/most_offered_services_map.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: figures/most_offered_services_map.png")

### 4.4 — Map 4: Least Offered Service

In [ ]:
least_colors = {
    1: "#D55E00", 2: "#56B4E9", 3: "#009E73", 4: "#F0E442", 5: "#0072B2",
    6: "#E69F00", 7: "#CC79A7", 8: "#F7C6C7", 9: "#999999", 10: "#A9A9A9",
    11: "#5D6D7E", 12: "#85C1E9", 13: "#F1948A", 14: "#BB8FCE", 15: "#16A085",
    16: "#F39C12", 17: "#2C3E50", 18: "#9B59B6", 19: "#E74C3C", 20: "#1ABC9C",
    21: "#34495E", 22: "#D4AC0D"
}

geo_l93["least_color"] = geo_l93["least_treated_service_id"].map(least_colors)

fig, ax = plt.subplots(figsize=(10, 12))

geo_l93.plot(
    color=geo_l93["least_color"],
    linewidth=0.5,
    edgecolor="black",
    ax=ax
)

# Annotate major cities using Lambert 93 centroids
for _, row in big_cities.iterrows():
    ax.scatter(row["centroid_x"], row["centroid_y"], color="white", edgecolor="black", s=60, zorder=3)
    ax.text(
        row["centroid_x"] + 3000, row["centroid_y"] + 3000,
        textwrap.fill(str(row["big_city_name"]), width=10),
        fontsize=8, fontweight="bold", color="black",
        ha="center", va="center",
        bbox=dict(facecolor="white", alpha=0.7, edgecolor="none", boxstyle="round,pad=0.3")
    )

# Legend — highlight key underserved conditions
highlight = {1: "Sex Addiction", 2: "Autism", 6: "Anorexia", 9: "Burnout", 21: "Anxiety"}
legend_patches = [
    mpatches.Patch(color=least_colors[sid], label=sname)
    for sid, sname in highlight.items()
]
ax.legend(handles=legend_patches, title="Least Offered Services", loc="lower left", fontsize=8)

ax.axis("off")

plt.tight_layout()
plt.savefig("figures/least_offered_services_map.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: figures/least_offered_services_map.png")

### 4.5 — Barplot: Frequency of Least Offered Services Across Departments

The map shows spatial distribution, but a barplot more clearly communicates which services are underserved most broadly (across many departments). Self-Assertion tops the list, followed by Sex Addiction and Practice Analysis.

In [ ]:
grouped = (
    geo_l93
    .groupby("least_treated_service_en", as_index=False)
    .agg(
        count=("least_treated_service_en", "count"),
        service_id=("least_treated_service_id", "first")
    )
)
grouped["pct"] = grouped["count"] / grouped["count"].sum() * 100
grouped["color"] = grouped["service_id"].map(least_colors)
grouped.sort_values("pct", ascending=False, inplace=True)

plt.figure(figsize=(12, 7))
sns.barplot(
    x="pct", y="least_treated_service_en",
    data=grouped,
    palette=grouped["color"].values
)

for i, (pct, count) in enumerate(zip(grouped["pct"], grouped["count"])):
    plt.text(pct + 0.4, i, f"{pct:.1f}% ({count})", va="center", fontsize=10)

plt.xlim(0, 25)
plt.xlabel("Percentage of Departments Where Service is Least Offered", fontsize=12)
plt.ylabel("Least Offered Services", fontsize=12)
plt.tight_layout()
plt.savefig("figures/least_offered_services_barplot.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: figures/least_offered_services_barplot.png")